In [4]:
import os
import re
import time
import shutil
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
from datetime import datetime, timedelta
from typing import List
import openpyxl
from python_calamine import CalamineWorkbook

In [5]:
first_glob = os.path.expanduser("~").replace("\\", "/")
test_path = f"{first_glob}/Concentrix Corporation"
if not os.path.exists(test_path):
    raise FileNotFoundError(f"SharePoint root folder not found: {test_path}")

folder_paths = {
    "hc_staffing":        f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Master Database - 2026.xlsx",
    "hc_active":          f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Master Database - 2026.xlsx",
    "hc_inactive":        f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Master Database - 2026.xlsx",
    "date_format":        f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Master Database - 2026.xlsx",
    "hc_extend_by_month": f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month/",
    "resources":          f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources",
    "team_alignment_wow": f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_TEAM_ALIGNMENT",
}

In [6]:
from datetime import datetime, timedelta, date as date_type

def normalize_date_col(series: pl.Series) -> pl.Series:
    EXCEL_EPOCH = datetime(1899, 12, 30)

    def _normalize(val):
        if val is None:
            return None
        # Calamine already parsed it as datetime/date object
        if isinstance(val, datetime):
            return val.date()
        if isinstance(val, date_type):
            return val

        s = str(val).strip()
        if not s or s.lower() in ("none", "nan", "null", ""):
            return None

        # Excel serial number (float/int stored as string)
        try:
            num = float(s)
            if 1 < num < 60000:
                return (EXCEL_EPOCH + timedelta(days=num)).date()
        except ValueError:
            pass

        # String date/datetime formats
        for fmt in (
            "%m/%d/%Y %H:%M",
            "%m/%d/%Y",
            "%Y-%m-%d %H:%M:%S",
            "%Y-%m-%d %H:%M",
            "%Y-%m-%d",
        ):
            try:
                return datetime.strptime(s[:len(fmt)], fmt).date()
            except ValueError:
                try:
                    return datetime.strptime(s, fmt).date()
                except ValueError:
                    continue
        return None

    parsed = [_normalize(v) for v in series.to_list()]
    return pl.Series(series.name, parsed, dtype=pl.Date)

def resolve_sheet_name(file_path: Path, keyword: str) -> str:
    wb = CalamineWorkbook.from_path(str(file_path))
    matches = [name for name in wb.sheet_names if keyword.lower() in name.lower()]
    if not matches:
        raise ValueError(f"No sheet containing '{keyword}' found in {file_path.name}. "
                         f"Available: {wb.sheet_names}")
    return matches[0]

def extract_and_clean_sheet(file_path, sheet_name, header_col_name, msa_col):
    df_raw = pl.read_excel(source=file_path, sheet_name=sheet_name, engine="calamine", read_options={"header_row": None})
    first_col = df_raw.columns[0]
    first_col_values = df_raw[first_col].cast(pl.Utf8).to_list()
    header_indices = [i for i, val in enumerate(first_col_values) if val is not None and str(val).strip() == header_col_name]
    header_idx = header_indices[0] if header_indices else 0
    df_data = df_raw.slice(header_idx)
    headers = df_data.row(0)
    seen = {}
    new_cols = []
    for i, h in enumerate(headers):
        col = str(h).strip() if h is not None else f"unnamed_{i}"
        if col in seen:
            seen[col] += 1
            col = f"{col}_{seen[col]}"
        else:
            seen[col] = 0
        new_cols.append(col)

    df_cleaned = df_data.slice(1)
    df_cleaned.columns = new_cols
    id_col = new_cols[0]
    df_cleaned = df_cleaned.with_columns(pl.col(id_col).cast(pl.Int64, strict=False))
    return df_cleaned.filter(pl.col(msa_col).str.to_lowercase().str.contains("expedia"))

def process_workday_headcount(input_file, output_dir):
    file_path = Path(input_file)
    out_dir   = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Reading Workday Dump: {file_path.name} ...")

    df_master = extract_and_clean_sheet(file_path, "Employee Master", "EMPLOYEE_NUMBER", "MSA Client")

    term_sheet = resolve_sheet_name(file_path, "Termination")
    df_term = extract_and_clean_sheet(file_path, term_sheet, "EMPLOYEE_ID", "MSA")

    df_term = df_term.with_columns(
        normalize_date_col(df_term["TERMINATION DATE"]).alias("Parsed_Date")
    ).with_columns(
        pl.col("Parsed_Date").dt.truncate("1mo").alias("Month_Start")
    )

    df_trans = extract_and_clean_sheet(file_path, "Transfer", "Employee ID", "MSA Client Current")
    df_trans = df_trans.with_columns(
        normalize_date_col(df_trans["Effective Date"]).alias("Parsed_Date")
    ).with_columns(
        pl.col("Parsed_Date").dt.truncate("1mo").alias("Month_Start")
    )

    months_term  = df_term.select("Month_Start").drop_nulls().unique().to_series().to_list()
    months_trans = df_trans.select("Month_Start").drop_nulls().unique().to_series().to_list()
    all_months   = sorted(set(months_term + months_trans))
    print(f"Found {len(all_months)} distinct months.")

    for month in all_months:
        month_str    = month.strftime("%Y_%m")
        out_filename = out_dir / f"WD_{month_str}.xlsx"
        df_term_m  = df_term.filter(pl.col("Month_Start") == month).drop(["Parsed_Date", "Month_Start"])
        df_trans_m = df_trans.filter(pl.col("Month_Start") == month).drop(["Parsed_Date", "Month_Start"])
        with pd.ExcelWriter(out_filename, engine="xlsxwriter") as writer:
            df_master.to_pandas().to_excel(writer, sheet_name="Employee Master", index=False)
            df_term_m.to_pandas().to_excel(writer,  sheet_name="Termination",     index=False)
            df_trans_m.to_pandas().to_excel(writer, sheet_name="Transfer",        index=False)
        print(f"  Generated: {out_filename.name}")
    print("Done.")


def generate_combined_headcount_csv():
    input_file = Path(f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/Workday Dump_Till_Date.xlsb")
    out_dir    = Path(f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/")
    out_dir.mkdir(parents=True, exist_ok=True)
    output_csv = out_dir / "Combined_WD_Data.csv"

    df_master = extract_and_clean_sheet(input_file, "Employee Master", "EMPLOYEE_NUMBER", "MSA Client").select([
        "EMPLOYEE_NUMBER", "FULL_NAME", "ORIGINAL_DATE_OF_HIRE", "WORKER_CATEGORY",
        "Email - Work", "Job Title", "Compensation Grade", "SUPERVISOR_ID",
        "SUPERVISOR_FULL_NAME", "SUPERVISOR_EMAIL_ID", "Employee Status",
        "Contract End Date", "Fixed Term Hire End Date",
        "MANAGER_02_ID", "MANAGER_02_FULL_NAME", "MANAGER_02_EMAIL_ID", "JOB_FUNCTION_DESCRIPTION",
    ])

    term_sheet = resolve_sheet_name(input_file, "Termination")
    df_term = extract_and_clean_sheet(input_file, term_sheet, "EMPLOYEE_ID", "MSA").select([
        pl.col("EMPLOYEE_ID").alias("EMPLOYEE_NUMBER"), "FULL_NAME",
        pl.col("EMAIL_ADDRESS").alias("Email - Work"),
        pl.col("ORIGINAL_HIRE_DATE").alias("ORIGINAL_DATE_OF_HIRE"),
        "END EMPLOYMENT DATE", "Termination Date", "Eligible for Rehire",
        "LWD", "TERMINATION DATE", "Termination Reason", "Resignation Reason",
        "WORKER_CATEGORY", pl.col("JOB_TITLE").alias("Job Title"),
        "Compensation Grade", "SUPERVISOR_ID", "SUPERVISOR_FULL_NAME", "SUPERVISOR_EMAIL_ID",
        pl.col("EMPLOYEE STATUS").alias("Employee Status"), "Contract End Date",
        pl.col("JOB_FUNCTION").alias("JOB_FUNCTION_DESCRIPTION"),
    ])

    df_combined = pl.concat([df_master, df_term], how="diagonal")

    date_cols = [
        "ORIGINAL_DATE_OF_HIRE", "END EMPLOYMENT DATE", "LWD",
        "Termination Date", "TERMINATION DATE", "Contract End Date", "Fixed Term Hire End Date",
    ]
    for col_name in date_cols:
        if col_name in df_combined.columns:
            df_combined = df_combined.with_columns(
                normalize_date_col(df_combined[col_name])
            )

    df_combined = df_combined.with_columns([
        pl.col("EMPLOYEE_NUMBER").cast(pl.Int64, strict=False),
        pl.col("SUPERVISOR_ID").cast(pl.Int64, strict=False),
        pl.col("MANAGER_02_ID").cast(pl.Int64, strict=False),
    ])
    df_combined.write_csv(output_csv)
    print(f"Combined CSV written to: {output_csv}")

if __name__ == "__main__":
    INPUT_FILE       = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/Workday Dump_Till_Date.xlsb"
    OUTPUT_DIRECTORY = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/WD"
    process_workday_headcount(INPUT_FILE, OUTPUT_DIRECTORY)
    generate_combined_headcount_csv()

Reading Workday Dump: Workday Dump_Till_Date.xlsb ...
Found 33 distinct months.
  Generated: WD_2023_11.xlsx
  Generated: WD_2023_12.xlsx
  Generated: WD_2024_01.xlsx
  Generated: WD_2024_02.xlsx
  Generated: WD_2024_03.xlsx
  Generated: WD_2024_04.xlsx
  Generated: WD_2024_05.xlsx
  Generated: WD_2024_06.xlsx
  Generated: WD_2024_07.xlsx
  Generated: WD_2024_08.xlsx
  Generated: WD_2024_09.xlsx
  Generated: WD_2024_10.xlsx
  Generated: WD_2024_11.xlsx
  Generated: WD_2024_12.xlsx
  Generated: WD_2025_01.xlsx
  Generated: WD_2025_02.xlsx
  Generated: WD_2025_03.xlsx
  Generated: WD_2025_04.xlsx
  Generated: WD_2025_05.xlsx
  Generated: WD_2025_06.xlsx
  Generated: WD_2025_07.xlsx
  Generated: WD_2025_08.xlsx
  Generated: WD_2025_09.xlsx
  Generated: WD_2025_10.xlsx
  Generated: WD_2025_11.xlsx
  Generated: WD_2025_12.xlsx
  Generated: WD_2026_01.xlsx
  Generated: WD_2026_02.xlsx
  Generated: WD_2026_03.xlsx
  Generated: WD_2026_04.xlsx
  Generated: WD_2026_05.xlsx
  Generated: WD_2026_

In [4]:
# 1/10 read sheets with data_only=True so formula cells return cached values
def read_excel_data_only(path, sheet_name):
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    ws = wb[sheet_name]
    rows = ws.values
    headers = [str(h).strip() if h is not None else f"col_{i}" for i, h in enumerate(next(rows))]
    df = pd.DataFrame(rows, columns=headers)
    wb.close()
    return df

hc_active        = read_excel_data_only(folder_paths["hc_active"],   "Active")
hc_inactive      = read_excel_data_only(folder_paths["hc_inactive"], "Inactive")
staffing_records = read_excel_data_only(folder_paths["hc_staffing"], "Staffing_Records")
date_format      = read_excel_data_only(folder_paths["date_format"], "Date_Format")

# 2/10 select columns from Active sheet
hc_active = hc_active[[
    "Site", "Queue Group", "CSG Joining Date", "Location", "People ID",
    "IEX ID", "OracleID", "Employee Name", "Alias", "Gender",
    "Contract Start Date", "Contract End Date", "Designation", "Grade", "LOB",
    "Role", "Multiple Chat Effective Date", "Primary role",
    "Secondary role", "TL ID", "Supervisor Name", "Supervisor Email",
    "Manager/OM Name", "Email Id", "Wave", "CCT Training",
    "Lodging - Training start date", "Lodging - Training end date",
    "Lodging - Nesting start date", "Lodging - Nesting end date",
    "Lodging - Certification Date (Original)",
    "Lodging - Certification Date (Actual)", "Lodging - Certification status",
    "Non-Lodging - Training start date", "Non-Lodging - Training end date",
    "Non-Lodging - Nesting start date", "Non-Lodging - Nesting end date",
    "Non-Lodging - Certification Date (Original)",
    "Non-Lodging - Certification Date (Actual)", "Non-Lodging - Certification status",
    "Production Date", "AON", "Tenure",
    "Current Queue Production Date", "Current Queue Production Duration", "Current Queue Tenure",
    "Nationality", "MSA", "Current Console Queue",
]]

# 3/10 select columns from Inactive — rename typos/differences to match Active
hc_inactive = hc_inactive[[
    "Site", "Queue Group", "CSG Joining Data", "Location", "People ID",
    "IEX ID", "OracleID", "Employee Name", "Alias", "Gender",
    "Contract Start Date", "Contract End Date", "Designation", "Grade", "LOB",
    "Role", "Multiple Chat Effective Date", "Primary role",
    "Secondary role", "TL ID", "Supervisor Name", "Supervisor Email",
    "Manager Name",
    "Email Id", "Wave", "CCT Training",
    "Lodging - Training start date", "Lodging - Training end date",
    "Lodging - Nesting start date", "Lodging - Nesting end date",
    "Lodging - Certification Date (Original)",
    "Lodging - Certification Date (Actual)", "Lodging - Certification status",
    "Non-Lodging - Training start date", "Non-Lodging - Training end date",
    "Non-Lodging - Nesting start date", "Non-Lodging - Nesting end date",
    "Non-Lodging - Certification Date (Original)",
    "Non-Lodging - Certification Date (Actual)", "Non-Lodging - Certification status",
    "Production Date", "AON", "Tenure",
    "Current Queue Production Date", "Current Queue Production Duration", "Current Queue Tenure",
    "Nationality", "MSA",
    "Current Console Queue", "LWD/Movement", "Attrition Type",
    "Reason for Attrition", "Month", "Week Ending",
]].rename(columns={
    "CSG Joining Data": "CSG Joining Date",
    "Manager Name":     "Manager/OM Name",
})

# 4/10 combine Active + Inactive
hc_combined = pd.concat([hc_active, hc_inactive], ignore_index=True)

# 5/10 build date spine 2023-12-01 to today+45d, cross-join with OracleID via Polars
start_date = datetime(2023, 12, 1)
end_date   = datetime.now() + timedelta(days=45)
date_list  = pd.date_range(start_date, end_date)
oracle_ids = hc_combined["OracleID"].dropna().unique().tolist()
pl_dates   = pl.Series("Date", date_list.to_pydatetime())
pl_ids     = pl.Series("OracleID", oracle_ids)
expanded_hc = (
    pl.DataFrame({"OracleID": pl_ids})
      .join(pl.DataFrame({"Date": pl_dates}), how="cross")
      .to_pandas()
)

# 6/10 attach employee attributes
HC_Data_Edition = expanded_hc.merge(hc_combined, on="OracleID", how="left")

# 7/10 normalize date columns
date_columns = [
    "Contract Start Date", "Contract End Date", "Multiple Chat Effective Date",
    "CCT Training",
    "Lodging - Training start date", "Lodging - Training end date",
    "Lodging - Nesting start date",  "Lodging - Nesting end date",
    "Lodging - Certification Date (Original)", "Lodging - Certification Date (Actual)",
    "Non-Lodging - Training start date", "Non-Lodging - Training end date",
    "Non-Lodging - Nesting start date",  "Non-Lodging - Nesting end date",
    "Non-Lodging - Certification Date (Original)", "Non-Lodging - Certification Date (Actual)",
    "Production Date", "LWD/Movement", "CSG Joining Date",
    "Current Queue Production Date",
]

for column in date_columns:
    HC_Data_Edition[column] = pd.to_datetime(HC_Data_Edition[column], errors="coerce")

# 8/10 remove nulls and duplicates
HC_Data_Edition.fillna(pd.NA, inplace=True)
HC_Data_Edition = HC_Data_Edition.sort_values(["OracleID", "Date"])
HC_Data_Edition = HC_Data_Edition.drop_duplicates(subset=["Date", "OracleID"])
HC_Data_Edition = HC_Data_Edition[HC_Data_Edition["OracleID"].notna()]

# 9/10 verify
print("Column check:", HC_Data_Edition.columns.tolist())

# 10/10 debug missing Employee Name
print(HC_Data_Edition[HC_Data_Edition["Employee Name"].isna()][["Date", "OracleID", "IEX ID", "Employee Name"]].head(20))
HC_Data_Edition


C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_28004\2502601213.py:98: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  HC_Data_Edition[column] = pd.to_datetime(HC_Data_Edition[column], errors="coerce")
C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_28004\2502601213.py:98: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  HC_Data_Edition[column] = pd.to_datetime(HC_Data_Edition[column], errors="coerce")


Column check: ['OracleID', 'Date', 'Site', 'Queue Group', 'CSG Joining Date', 'Location', 'People ID', 'IEX ID', 'Employee Name', 'Alias', 'Gender', 'Contract Start Date', 'Contract End Date', 'Designation', 'Grade', 'LOB', 'Role', 'Multiple Chat Effective Date', 'Primary role', 'Secondary role', 'TL ID', 'Supervisor Name', 'Supervisor Email', 'Manager/OM Name', 'Email Id', 'Wave', 'CCT Training', 'Lodging - Training start date', 'Lodging - Training end date', 'Lodging - Nesting start date', 'Lodging - Nesting end date', 'Lodging - Certification Date (Original)', 'Lodging - Certification Date (Actual)', 'Lodging - Certification status', 'Non-Lodging - Training start date', 'Non-Lodging - Training end date', 'Non-Lodging - Nesting start date', 'Non-Lodging - Nesting end date', 'Non-Lodging - Certification Date (Original)', 'Non-Lodging - Certification Date (Actual)', 'Non-Lodging - Certification status', 'Production Date', 'AON', 'Tenure', 'Current Queue Production Date', 'Current Queue

,OracleID,Date,Site,Queue Group,CSG Joining Date,Location,People ID,IEX ID,Employee Name,Alias,...,Current Queue Production Duration,Current Queue Tenure,Nationality,MSA,Current Console Queue,LWD/Movement,Attrition Type,Reason for Attrition,Month,Week Ending
258315,1081503.0,2023-12-01 00:00:00,-,-,NaT,Vietnam,178879477,<NA>,TRAN PHAM NGOC VAN,Van,...,-,-,Vietnamese,Expedia,-,2024-05-02,Movement,Movement to Etraveli,May'24,2024-05-05
258316,1081503.0,2023-12-02 00:00:00,-,-,NaT,Vietnam,178879477,<NA>,TRAN PHAM NGOC VAN,Van,...,-,-,Vietnamese,Expedia,-,2024-05-02,Movement,Movement to Etraveli,May'24,2024-05-05
258317,1081503.0,2023-12-03 00:00:00,-,-,NaT,Vietnam,178879477,<NA>,TRAN PHAM NGOC VAN,Van,...,-,-,Vietnamese,Expedia,-,2024-05-02,Movement,Movement to Etraveli,May'24,2024-05-05
258318,1081503.0,2023-12-04 00:00:00,-,-,NaT,Vietnam,178879477,<NA>,TRAN PHAM NGOC VAN,Van,...,-,-,Vietnamese,Expedia,-,2024-05-02,Movement,Movement to Etraveli,May'24,2024-05-05
258319,1081503.0,2023-12-05 00:00:00,-,-,NaT,Vietnam,178879477,<NA>,TRAN PHAM NGOC VAN,Van,...,-,-,Vietnamese,Expedia,-,2024-05-02,Movement,Movement to Etraveli,May'24,2024-05-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151945,103612128.0,2026-09-04 00:00:00,ONEHUB,-,NaT,Vietnam,<NA>,<NA>,TRAN GIA LONG,<NA>,...,-,-,Vietnamese,Expedia,-,NaT,<NA>,<NA>,<NA>,NaT
151946,103612128.0,2026-09-05 00:00:00,ONEHUB,-,NaT,Vietnam,<NA>,<NA>,TRAN GIA LONG,<NA>,...,-,-,Vietnamese,Expedia,-,NaT,<NA>,<NA>,<NA>,NaT
151947,103612128.0,2026-09-06 00:00:00,ONEHUB,-,NaT,Vietnam,<NA>,<NA>,TRAN GIA LONG,<NA>,...,-,-,Vietnamese,Expedia,-,NaT,<NA>,<NA>,<NA>,NaT
151948,103612128.0,2026-09-07 00:00:00,ONEHUB,-,NaT,Vietnam,<NA>,<NA>,TRAN GIA LONG,<NA>,...,-,-,Vietnamese,Expedia,-,NaT,<NA>,<NA>,<NA>,NaT


In [5]:
# 1/10 normalize dates
HC_Data_Edition["Date"]           = pd.to_datetime(HC_Data_Edition["Date"],            errors="coerce")
staffing_records["Effective Date"] = pd.to_datetime(staffing_records["Effective Date"], errors="coerce")

# 2/10 unique (Date, OracleID) pairs
HC_Data_Edition_Only_ID = HC_Data_Edition[["Date", "OracleID"]]
staffing_records_date   = staffing_records[["Effective Date", "OracleID"]].rename(columns={"OracleID": "Changed.OracleID"})

# 3/10 keep only OracleIDs present in Staffing_Records
merged_id = pd.merge(HC_Data_Edition_Only_ID, staffing_records_date[["Changed.OracleID"]], how="left", left_on="OracleID", right_on="Changed.OracleID")
merged_id = merged_id[merged_id["Changed.OracleID"].notna()].drop(columns=["Changed.OracleID"])

# 4/10 join on (Date, OracleID) to find staffing changes
merged_effective_date = pd.merge(merged_id, staffing_records_date[["Effective Date", "Changed.OracleID"]], how="left", left_on=["Date", "OracleID"], right_on=["Effective Date", "Changed.OracleID"])
merged_effective_date = merged_effective_date.sort_values(["OracleID", "Date"]).drop_duplicates(subset=["Date", "OracleID", "Effective Date", "Changed.OracleID"])

# 5/10 forward-fill Effective Date
merged_effective_date["Effective Date"] = merged_effective_date["Effective Date"].ffill()

# 6/10 enforce: Effective Date must not exceed snapshot Date
mask = merged_effective_date["Effective Date"] > merged_effective_date["Date"]
merged_effective_date.loc[mask, "Effective Date"] = pd.NaT
merged_effective_date = merged_effective_date[merged_effective_date["Effective Date"].notna()].drop(columns=["Changed.OracleID"])

# 7/10 attach full staffing record attributes
merged_staffing_records = pd.merge(merged_effective_date, staffing_records, how="left", on=["Effective Date", "OracleID"])

# 8/10 pull attrition columns
HC_Data_Edition_Inactive = HC_Data_Edition[["Date", "Site", "Queue Group", "CSG Joining Date", "OracleID", "LWD/Movement", "Attrition Type", "Reason for Attrition"]]
merged_inactive_cols = pd.merge(merged_staffing_records, HC_Data_Edition_Inactive, on=["Date", "OracleID"], how="left")
staffing_records = merged_inactive_cols

# 9/10 identify unchanged rows
merged_staffing_records = HC_Data_Edition.merge(staffing_records[["Date", "OracleID", "Effective Date"]], how="left", on=["Date", "OracleID"])
HC_Data_Edition_not_change = merged_staffing_records[merged_staffing_records["Effective Date"].isna()]

# 10/10 combine changed + unchanged
hc_extend = pd.concat([HC_Data_Edition_not_change, staffing_records], ignore_index=True)
hc_extend = hc_extend.drop(columns=["Effective Date"])


In [6]:
# 1/12 drop training end date cols (redundant — nesting start is the boundary)
hc_extend = hc_extend.drop(columns=["Lodging - Training end date", "Non-Lodging - Training end date"], errors="ignore")

# 2/12 rename to short names
hc_extend = hc_extend.rename(columns={
    "Lodging - Training start date":              "Lodging Training",
    "Lodging - Nesting start date":               "Lodging Nesting",
    "Lodging - Nesting end date":                 "Lodging Nesting End",
    "Lodging - Certification Date (Original)":    "Lodging Original",
    "Lodging - Certification Date (Actual)":      "Lodging Actual",
    "Lodging - Certification status":             "Lodging",
    "Non-Lodging - Training start date":          "NonLodging Training",
    "Non-Lodging - Nesting start date":           "NonLodging Nesting",
    "Non-Lodging - Nesting end date":             "NonLodging Nesting End",
    "Non-Lodging - Certification Date (Original)": "NonLodging Original",
    "Non-Lodging - Certification Date (Actual)":  "NonLodging Actual",
    "Non-Lodging - Certification status":         "NonLodging",
    "Production Date":                            "Production",
})

for col in ["Lodging Nesting End", "NonLodging Nesting End"]:
    if col in hc_extend.columns:
        hc_extend[col] = pd.to_datetime(hc_extend[col], errors="coerce")
        
# 3/12 parse date columns
date_columns = [
    "Date", "CCT Training",
    "Lodging Training", "Lodging Nesting", "Lodging Nesting End",
    "Lodging Original", "Lodging Actual",
    "NonLodging Training", "NonLodging Nesting", "NonLodging Nesting End",
    "NonLodging Original", "NonLodging Actual",
    "Production", "Multiple Chat Effective Date", "LWD/Movement", "CSG Joining Date",
]
for col in date_columns:
    if col in hc_extend.columns:
        hc_extend[col] = pd.to_datetime(hc_extend[col], errors="coerce")

# 4/12 Production = Certification Actual (no Monday offset)
hc_extend["Lodging Production"]    = pd.to_datetime(hc_extend["Lodging Actual"],    errors="coerce")
hc_extend["NonLodging Production"] = pd.to_datetime(hc_extend["NonLodging Actual"], errors="coerce")

# 5/12 Extended Nesting: Original < Actual means re-nesting period exists
lg_has_extend = hc_extend["Lodging Original"].notna() & hc_extend["Lodging Actual"].notna() & (hc_extend["Lodging Original"] < hc_extend["Lodging Actual"])
nl_has_extend = hc_extend["NonLodging Original"].notna() & hc_extend["NonLodging Actual"].notna() & (hc_extend["NonLodging Original"] < hc_extend["NonLodging Actual"])
hc_extend["Lodging Extended Nesting"]    = hc_extend["Lodging Original"].where(lg_has_extend)
hc_extend["NonLodging Extended Nesting"] = hc_extend["NonLodging Original"].where(nl_has_extend)

# 6/12 drop intermediate columns
hc_extend = hc_extend.drop(columns=[c for c in ["Lodging Original", "NonLodging Original", "Lodging Actual", "NonLodging Actual", "Lodging", "NonLodging", "Contract End Date"] if c in hc_extend.columns])

# 7/12 reorder columns
column_order = [
    "Date", "Site", "Queue Group", "OracleID", "People ID", "IEX ID",
    "Employee Name", "Alias", "Gender", "Designation", "Grade", "LOB",
    "Multiple Chat Effective Date", "Role", "Primary role", "Secondary role",
    "TL ID", "Supervisor Name", "Supervisor Email", "Manager/OM Name", "Email Id",
    "Wave", "CCT Training",
    "Lodging Training", "Lodging Nesting", "Lodging Nesting End",
    "Lodging Extended Nesting", "Lodging Production",
    "NonLodging Training", "NonLodging Nesting", "NonLodging Nesting End",
    "NonLodging Extended Nesting", "NonLodging Production",
    "Production", "Contract Start Date", "Attrition Type", "Reason for Attrition",
    "LWD/Movement", "CSG Joining Date", "Current Console Queue",
    "Current Queue Production Date", "Current Queue Production Duration", "Current Queue Tenure",
]
hc_extend = hc_extend[[c for c in column_order if c in hc_extend.columns]]
hc_extend = hc_extend.sort_values("Date")

# 8/12 reconfirm date dtypes
for col in ["Date", "CCT Training", "Lodging Training", "Lodging Nesting", "Lodging Nesting End", "Lodging Extended Nesting", "Lodging Production", "NonLodging Training", "NonLodging Nesting", "NonLodging Nesting End", "NonLodging Extended Nesting", "NonLodging Production"]:
    if col in hc_extend.columns:
        hc_extend[col] = pd.to_datetime(hc_extend[col], errors="coerce")

# 9/12 vectorised Detail Status
d     = hc_extend["Date"]
nl_p  = hc_extend["NonLodging Production"]
nl_ex = hc_extend["NonLodging Extended Nesting"]
nl_n  = hc_extend["NonLodging Nesting"]
nl_ne = hc_extend["NonLodging Nesting End"]
nl_t  = hc_extend["NonLodging Training"]
l_p   = hc_extend["Lodging Production"]
l_ex  = hc_extend["Lodging Extended Nesting"]
l_n   = hc_extend["Lodging Nesting"]
l_ne  = hc_extend["Lodging Nesting End"]
l_t   = hc_extend["Lodging Training"]
cct   = hc_extend["CCT Training"]

hc_extend["Detail Status"] = np.select(
    [
        d.notna() & nl_p.notna()  & (d >= nl_p),
        d.notna() & nl_ex.notna() & nl_p.isna()   & (d >= nl_ex),
        d.notna() & nl_ex.notna() & nl_p.notna()  & (nl_ex <= d) & (d < nl_p),
        d.notna() & nl_n.notna()  & nl_ne.notna() & nl_ex.isna() & (nl_n <= d) & (d <= nl_ne),
        d.notna() & nl_n.notna()  & nl_ne.isna()  & nl_ex.isna() & nl_p.isna()  & (d >= nl_n),
        d.notna() & nl_n.notna()  & nl_ne.isna()  & nl_ex.isna() & nl_p.notna() & (nl_n <= d) & (d < nl_p),
        d.notna() & nl_n.notna()  & nl_ne.notna() & nl_ex.notna() & (nl_n <= d) & (d < nl_ex),
        d.notna() & nl_t.notna()  & nl_n.isna()  & (d >= nl_t),
        d.notna() & nl_t.notna()  & nl_n.notna() & (nl_t <= d) & (d < nl_n),
        d.notna() & l_p.notna()   & nl_t.isna()  & (d >= l_p),
        d.notna() & l_p.notna()   & nl_t.notna() & (l_p <= d) & (d < nl_t),
        d.notna() & l_ex.notna()  & l_p.isna()   & (d >= l_ex),
        d.notna() & l_ex.notna()  & l_p.notna()  & (l_ex <= d) & (d < l_p),
        d.notna() & l_n.notna()   & l_ne.notna() & l_ex.isna() & (l_n <= d) & (d <= l_ne),
        d.notna() & l_n.notna()   & l_ne.isna()  & l_ex.isna() & l_p.isna()  & (d >= l_n),
        d.notna() & l_n.notna()   & l_ne.isna()  & l_ex.isna() & l_p.notna() & (l_n <= d) & (d < l_p),
        d.notna() & l_n.notna()   & l_ne.notna() & l_ex.notna() & (l_n <= d) & (d < l_ex),
        d.notna() & l_t.notna()   & l_n.isna()  & (d >= l_t),
        d.notna() & l_t.notna()   & l_n.notna() & (l_t <= d) & (d < l_n),
        d.notna() & cct.notna() & l_t.isna() & nl_t.isna() & (d >= cct),
        d.notna() & cct.notna() & l_t.isna() & nl_t.notna() & (cct <= d) & (d < nl_t),
        d.notna() & cct.notna() & l_t.notna() & (cct <= d) & (d < l_t),
        d.notna() & cct.notna() & (d < cct),
    ],
    [
        "Non Lodging Production",
        "Non Lodging Extended Nesting", "Non Lodging Extended Nesting",
        "Non Lodging Nesting", "Non Lodging Nesting", "Non Lodging Nesting", "Non Lodging Nesting",
        "Non Lodging Training", "Non Lodging Training",
        "Lodging Production", "Lodging Production",
        "Lodging Extended Nesting", "Lodging Extended Nesting",
        "Lodging Nesting", "Lodging Nesting", "Lodging Nesting", "Lodging Nesting",
        "Lodging Training", "Lodging Training",
        "CCT Training", "CCT Training", "CCT Training",
        "Unavailable",
    ],
    default="-",
)

# 10/12 AON
hc_extend["AON"] = np.where(hc_extend["Detail Status"] == "Lodging Production", (d - l_p).dt.days, np.where(hc_extend["Detail Status"] == "Non Lodging Production", (d - nl_p).dt.days, np.nan))
hc_extend["AON"] = pd.to_numeric(hc_extend["AON"], errors="coerce")

# 11/12 LG Tenure
aon = hc_extend["AON"]
lg_status = hc_extend["Detail Status"]
hc_extend["LG Tenure"] = np.select(
    [lg_status.isin(["Lodging Nesting", "Lodging Extended Nesting"]), l_p.notna() & (d >= l_p) & aon.between(0, 30), l_p.notna() & (d >= l_p) & aon.between(31, 60), l_p.notna() & (d >= l_p) & aon.between(61, 90), l_p.notna() & (d >= l_p) & (aon > 90)],
    ["Nesting", "0-30 Days", "30-60 Days", "60-90 Days", "> 90 Days"],
    default=None,
)

# 12/12 NL Tenure
hc_extend["NL Tenure"] = np.select(
    [lg_status.isin(["Non Lodging Nesting", "Non Lodging Extended Nesting"]), nl_p.notna() & (d >= nl_p) & aon.between(0, 30), nl_p.notna() & (d >= nl_p) & aon.between(31, 60), nl_p.notna() & (d >= nl_p) & aon.between(61, 90), nl_p.notna() & (d >= nl_p) & (aon > 90)],
    ["Nesting", "0-30 Days", "30-60 Days", "60-90 Days", "> 90 Days"],
    default=None,
)

print(hc_extend.columns.tolist())

['Date', 'Site', 'Queue Group', 'OracleID', 'People ID', 'IEX ID', 'Employee Name', 'Alias', 'Gender', 'Designation', 'Grade', 'LOB', 'Multiple Chat Effective Date', 'Role', 'Primary role', 'Secondary role', 'TL ID', 'Supervisor Name', 'Supervisor Email', 'Manager/OM Name', 'Email Id', 'Wave', 'CCT Training', 'Lodging Training', 'Lodging Nesting', 'Lodging Nesting End', 'Lodging Extended Nesting', 'Lodging Production', 'NonLodging Training', 'NonLodging Nesting', 'NonLodging Nesting End', 'NonLodging Extended Nesting', 'NonLodging Production', 'Production', 'Contract Start Date', 'Attrition Type', 'Reason for Attrition', 'LWD/Movement', 'CSG Joining Date', 'Current Console Queue', 'Current Queue Production Date', 'Current Queue Production Duration', 'Current Queue Tenure', 'Detail Status', 'AON', 'LG Tenure', 'NL Tenure']


In [7]:
# 1/15 Status
hc_extend["Status"] = np.where(hc_extend["LWD/Movement"].notna() & (hc_extend["Date"] >= hc_extend["LWD/Movement"]), np.where(hc_extend["Attrition Type"] == "Movement", "Transfered", "Terminated"), np.where(hc_extend["Date"] >= hc_extend["CCT Training"], "Active", None)).astype(object)

# 2/15 Concurrency
hc_extend["Concurrency"] = np.where(hc_extend["Detail Status"].str.contains("Production|Nesting", na=False), np.where(hc_extend["Multiple Chat Effective Date"].notna() & (hc_extend["Date"] >= hc_extend["Multiple Chat Effective Date"]), 2, 1), 0)

# 3/15 LOB_2
d_status = hc_extend["Detail Status"]
lob      = hc_extend["LOB"]
role     = hc_extend["Role"]
hc_extend["LOB_2"] = np.select(
    [
        d_status.str.contains("Production", na=False) & (lob == "Lodging")     & role.str.contains("Chat",  na=False),
        d_status.str.contains("Production", na=False) & (lob == "Non_Lodging") & role.str.contains("Chat",  na=False),
        d_status.isin(["Lodging Extended Nesting", "Lodging Nesting"])           & role.str.contains("Chat",  na=False),
        d_status.isin(["Non Lodging Extended Nesting", "Non Lodging Nesting"])   & role.str.contains("Chat",  na=False),
        d_status.str.contains("Production", na=False) & (lob == "Lodging")     & role.str.contains("Voice", na=False),
        d_status.str.contains("Production", na=False) & (lob == "Non_Lodging") & role.str.contains("Voice", na=False),
        d_status.isin(["Lodging Extended Nesting", "Lodging Nesting"])           & role.str.contains("Voice", na=False),
        d_status.isin(["Non Lodging Extended Nesting", "Non Lodging Nesting"])   & role.str.contains("Voice", na=False),
    ],
    ["Lodging Chat", "Non Lodging Chat", "Lodging Nesting Chat", "Non Lodging Nesting Chat", "Lodging Voice", "Non Lodging Voice", "Lodging Nesting Voice", "Non Lodging Nesting Voice"],
    default=None,
).astype(object)
hc_extend["LOB_3"] = hc_extend["LOB_2"]

# 4/15 sanitise NA
hc_extend[["Detail Status", "AON", "LG Tenure", "NL Tenure", "Concurrency", "Status"]] = hc_extend[["Detail Status", "AON", "LG Tenure", "NL Tenure", "Concurrency", "Status"]].replace({pd.NA: np.nan})

# 5/15 Performance_Calculation
hc_extend["Performance_Calculation"] = np.where(hc_extend["Detail Status"].isna() | hc_extend["Detail Status"].isin(["CCT Training", "Lodging Training", "Non Lodging Training", "Unavailable"]), "Excluded", "Included")

# 6/15 drop Unavailable rows
hc_extend = hc_extend[hc_extend["Detail Status"] != "Unavailable"]

# 7/15 deduplicate
hc_extend = hc_extend.drop_duplicates(subset=["Date", "IEX ID", "Employee Name"])

# 8/15 merge calendar metadata
hc_extend = pd.merge(hc_extend, date_format, on="Date", how="left")

# 9/15 Stage
hc_extend["Stage"] = np.select([hc_extend["Detail Status"].str.contains("Production", na=False), hc_extend["Detail Status"].str.contains("Training", na=False), hc_extend["Detail Status"].str.contains("Nesting", na=False)], ["In Production", "In Training", "In Nesting"], default=None)

# 10/15 ATT_Reason_LV3
hc_extend["ATT_Reason_LV3"] = hc_extend["Reason for Attrition"].str.split(" ").str[4:].str.join(" ")

# 11/15 HC Open/Close counters
lwd_mask = hc_extend["LWD/Movement"].isna() | (hc_extend["Date"] <= hc_extend["LWD/Movement"])
hc_extend["HC Open By Week"]    = np.where(lwd_mask & (hc_extend["Date"] == hc_extend["Date Start Week"]),  1, 0)
hc_extend["HC Closed By Week"]  = np.where(lwd_mask & (hc_extend["Date"] == hc_extend["Date End Week"]),    1, 0)
hc_extend["HC Open By Month"]   = np.where(lwd_mask & (hc_extend["Date"] == hc_extend["Date Start Month"]), 1, 0)
hc_extend["HC Closed By Month"] = np.where(lwd_mask & (hc_extend["Date"] == hc_extend["Date End Month"]),   1, 0)

# 12/15 ATT / Movement
hc_extend["ATT/Movement"] = np.where(hc_extend["Date"] == hc_extend["LWD/Movement"] + pd.Timedelta(days=1), 1, 0)
hc_extend["ATT"]      = np.where((hc_extend["ATT/Movement"] == 1) & hc_extend["Attrition Type"].isin(["Voluntary", "Involuntary"]), 1, 0)
hc_extend["Movement"] = hc_extend["ATT/Movement"] - hc_extend["ATT"]

# 13/15 fill NaN flags
for col in ["ATT/Movement", "ATT", "Movement"]:
    hc_extend[col] = hc_extend[col].fillna(0)

# 14/15 Active and CSG flags
hc_extend["Active"] = np.where(hc_extend["LWD/Movement"].isna(), "Yes", "No")
hc_extend["CSG"]    = np.where(hc_extend["CSG Joining Date"].isna() | (hc_extend["Date"] < hc_extend["CSG Joining Date"]), 0, 1)

# 15/15 final column order
column_order = [
    "Year", "Month", "Site", "Queue Group", "Week Begin",
    "Date Start Month", "Date End Month", "Date Start Week", "Date End Week",
    "Week", "Day", "Date", "OracleID", "People ID", "IEX ID", "Employee Name",
    "Alias", "Gender", "Designation", "Grade", "LOB", "Role",
    "Multiple Chat Effective Date", "Primary role", "Secondary role",
    "TL ID", "Supervisor Name", "Supervisor Email", "Manager/OM Name",
    "Email Id", "Wave", "CCT Training",
    "Lodging Training", "Lodging Nesting", "Lodging Nesting End",
    "Lodging Extended Nesting", "CSG Joining Date", "Lodging Production",
    "NonLodging Training", "NonLodging Nesting", "NonLodging Nesting End",
    "NonLodging Extended Nesting", "NonLodging Production",
    "Production", "Contract Start Date", "LWD/Movement",
    "Reason for Attrition", "Attrition Type", "ATT_Reason_LV3",
    "Detail Status", "AON", "LG Tenure", "NL Tenure", "Concurrency",
    "Status", "LOB_2", "LOB_3", "Performance_Calculation", "Stage",
    "HC Open By Week", "HC Closed By Week", "HC Open By Month", "HC Closed By Month",
    "ATT/Movement", "ATT", "Movement", "CSG", "Active", "Current Console Queue",
    "Current Queue Production Date", "Current Queue Production Duration", "Current Queue Tenure",
]
hc_extend = hc_extend[[c for c in column_order if c in hc_extend.columns]]
hc_extend.reset_index(drop=True, inplace=True)


In [8]:
hc_extend['Date']


0        2023-12-01
1        2023-12-01
2        2023-12-01
3        2023-12-01
4        2023-12-01
            ...    
627779   2026-09-08
627780   2026-09-08
627781   2026-09-08
627782   2026-09-08
627783   2026-09-08
Name: Date, Length: 627784, dtype: datetime64[ns]

In [9]:
# 1/9 read Mini Team sheet
mini_team = read_excel_data_only(folder_paths["hc_staffing"], "Mini Team")
mini_team = mini_team[["Mini TL", "Mini TL - Email", "Emp Email", "Role", "Active", "Mini TL - Short Name", "Effective Date"]].rename(columns={"Emp Email": "Email Id"})

# 2/9 normalize dates
mini_team["Effective Date"] = pd.to_datetime(mini_team["Effective Date"], errors="coerce")
hc_extend["Date"]           = pd.to_datetime(hc_extend["Date"],           errors="coerce")

# 3/9 unique (Date, Email Id) pairs
hc_extend_email_date = hc_extend[["Date", "Email Id"]].drop_duplicates()

# 4/9 join to get all effective dates per email
merge_key = pd.merge(hc_extend_email_date, mini_team[["Email Id", "Effective Date"]], on="Email Id", how="left")

# 5/9 keep only effective dates <= snapshot date
merge_key = merge_key[merge_key["Effective Date"] <= merge_key["Date"]]

# 6/9 retain latest effective date per (Email Id, Date)
merge_key = merge_key.sort_values(["Email Id", "Date", "Effective Date"]).drop_duplicates(subset=["Email Id", "Date"], keep="last")

# 7/9 attach Mini TL details
merge_mini_team = pd.merge(merge_key, mini_team[["Email Id", "Effective Date", "Mini TL - Email", "Mini TL - Short Name"]], on=["Email Id", "Effective Date"], how="left").sort_values("Date")

# 8/9 join back onto hc_extend
hc_extend = pd.merge(hc_extend, merge_mini_team[["Email Id", "Date", "Mini TL - Email", "Mini TL - Short Name", "Effective Date"]], on=["Email Id", "Date"], how="left")

# 9/9 rename Effective Date
hc_extend = hc_extend.rename(columns={"Effective Date": "Mini TL Start Date"})
hc_extend


,Year,Month,Site,Queue Group,Week Begin,Date Start Month,Date End Month,Date Start Week,Date End Week,Week,...,Movement,CSG,Active,Current Console Queue,Current Queue Production Date,Current Queue Production Duration,Current Queue Tenure,Mini TL - Email,Mini TL - Short Name,Mini TL Start Date
0,2023,Dec-23,-,-,WB2711,2023-12-01,2023-12-31,2023-11-27,2023-12-03,49,...,0,0,No,-,NaT,-,-,NaN,NaN,NaT
1,2023,Dec-23,FLEMINGTON,-,WB2711,2023-12-01,2023-12-31,2023-11-27,2023-12-03,49,...,0,0,No,-,NaT,-,-,NaN,NaN,NaT
2,2023,Dec-23,FLEMINGTON,-,WB2711,2023-12-01,2023-12-31,2023-11-27,2023-12-03,49,...,0,0,No,-,NaT,-,-,NaN,NaN,NaT
3,2023,Dec-23,-,-,WB2711,2023-12-01,2023-12-31,2023-11-27,2023-12-03,49,...,0,0,No,-,NaT,-,-,NaN,NaN,NaT
4,2023,Dec-23,ONEHUB,-,WB2711,2023-12-01,2023-12-31,2023-11-27,2023-12-03,49,...,0,0,No,-,2024-01-01 00:00:00,573,> 3 Months,NaN,NaN,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
627839,2026,Sep-26,QTSC,-,WB0709,2026-09-01,2026-09-30,2026-09-07,2026-09-13,37,...,0,0,No,Chat_OD_EN_Lodging,NaT,-,-,NaN,NaN,NaT
627840,2026,Sep-26,ONEHUB,CSG,WB0709,2026-09-01,2026-09-30,2026-09-07,2026-09-13,37,...,0,1,No,Chat_OD_EN_Car_Activity,-,-,-,hoangmyanh.tran@concentrix.com,Mya,2025-02-01
627841,2026,Sep-26,ONEHUB,-,WB0709,2026-09-01,2026-09-30,2026-09-07,2026-09-13,37,...,0,0,No,Chat_OD_EN_Dual_GDS,2025-02-02 00:00:00,79,60-90 Days,thithuminh.le@concentrix.com,Mia,2025-02-01
627842,2026,Sep-26,ONEHUB,CSG,WB0709,2026-09-01,2026-09-30,2026-09-07,2026-09-13,37,...,0,1,No,Chat_OD_EN_Car_Activity,-,-,-,NaN,NaN,NaT


In [10]:
dtype_map = {
    "Year": "Int64", "Month": "string", "Site": "string", "Queue Group": "string",
    "Week Begin": "string", "Week": "Int64", "Day": "string",
    "OracleID": "Int64", "People ID": "Int64", "IEX ID": "Int64",
    "Employee Name": "string", "Alias": "string", "Gender": "string",
    "Designation": "string", "Grade": "string", "LOB": "string",
    "Role": "string", "Primary role": "string", "Secondary role": "string",
    "TL ID": "Int64", "Supervisor Name": "string", "Supervisor Email": "string",
    "Manager/OM Name": "string", "Email Id": "string",
    "Reason for Attrition": "string", "Attrition Type": "string",
    "ATT_Reason_LV3": "string", "Detail Status": "string",
    "LG Tenure": "string", "NL Tenure": "string", "Status": "string",
    "LOB_2": "string", "LOB_3": "string",
    "Performance_Calculation": "string", "Stage": "string",
    "Wave": "object",
    "AON": "Int64", "Concurrency": "Int64",
    "HC Open By Week": "Int64", "HC Closed By Week": "Int64",
    "HC Open By Month": "Int64", "HC Closed By Month": "Int64",
    "ATT/Movement": "Int64", "ATT": "Int64", "Movement": "Int64",
    "CSG": "Int64", "Active": "string",
    "Mini TL - Email": "string", "Mini TL - Short Name": "string",
    "Current Console Queue": "string",
    "Current Queue Production Duration": "string", "Current Queue Tenure": "string",
}

date_cols = [
    "Date Start Month", "Date End Month", "Date Start Week", "Date End Week",
    "Date", "Multiple Chat Effective Date", "CCT Training",
    "Lodging Training", "Lodging Nesting", "Lodging Nesting End",
    "Lodging Extended Nesting", "CSG Joining Date", "Lodging Production",
    "NonLodging Training", "NonLodging Nesting", "NonLodging Nesting End",
    "NonLodging Extended Nesting", "NonLodging Production",
    "Production", "Contract Start Date", "LWD/Movement", "Mini TL Start Date",
    "Current Queue Production Date",
]

for col in date_cols:
    if col in hc_extend.columns:
        hc_extend[col] = pd.to_datetime(hc_extend[col], errors="coerce").dt.date

for col, dtype in dtype_map.items():
    if col not in hc_extend.columns:
        continue
    if dtype == "Int64":
        hc_extend[col] = pd.to_numeric(hc_extend[col], errors="coerce").astype("Int64")
    elif dtype == "string":
        hc_extend[col] = hc_extend[col].astype("string")
    else:
        hc_extend[col] = hc_extend[col].astype(dtype)

print(hc_extend.dtypes)

Year                                          Int64
Month                                string[python]
Site                                 string[python]
Queue Group                          string[python]
Week Begin                           string[python]
                                          ...      
Current Queue Production Duration    string[python]
Current Queue Tenure                 string[python]
Mini TL - Email                      string[python]
Mini TL - Short Name                 string[python]
Mini TL Start Date                           object
Length: 75, dtype: object


In [11]:
print(hc_extend.columns)

Index(['Year', 'Month', 'Site', 'Queue Group', 'Week Begin',
       'Date Start Month', 'Date End Month', 'Date Start Week',
       'Date End Week', 'Week', 'Day', 'Date', 'OracleID', 'People ID',
       'IEX ID', 'Employee Name', 'Alias', 'Gender', 'Designation', 'Grade',
       'LOB', 'Role', 'Multiple Chat Effective Date', 'Primary role',
       'Secondary role', 'TL ID', 'Supervisor Name', 'Supervisor Email',
       'Manager/OM Name', 'Email Id', 'Wave', 'CCT Training',
       'Lodging Training', 'Lodging Nesting', 'Lodging Nesting End',
       'Lodging Extended Nesting', 'CSG Joining Date', 'Lodging Production',
       'NonLodging Training', 'NonLodging Nesting', 'NonLodging Nesting End',
       'NonLodging Extended Nesting', 'NonLodging Production', 'Production',
       'Contract Start Date', 'LWD/Movement', 'Reason for Attrition',
       'Attrition Type', 'ATT_Reason_LV3', 'Detail Status', 'AON', 'LG Tenure',
       'NL Tenure', 'Concurrency', 'Status', 'LOB_2', 'LOB_3',
       

In [12]:
hc_extend["Date"]            = pd.to_datetime(hc_extend["Date"],            errors="coerce")
hc_extend["Date Start Week"] = pd.to_datetime(hc_extend["Date Start Week"], errors="coerce")

hc_weekly_lob = hc_extend[
    (hc_extend["Year"] == 2026)
    & hc_extend["Date"].dt.weekday.eq(0)
    & hc_extend["Designation"].astype(str).str.contains("Advisor I, Customer Service", na=False)
    & hc_extend["LOB"].astype(str).str.contains("Lodging|LG|NL", na=False, regex=True)
    & (hc_extend["Status"].astype(str) == "Active")
].copy()

hc_weekly_lob["LOB_Group"] = np.where(
    hc_weekly_lob["LOB"].astype(str).str.contains("Non_Lodging|_NL", na=False, regex=True),
    "NL",
    "LG"
)

hc_weekly_lob["LOB_Detail"] = np.select(
    [
        hc_weekly_lob["LOB"].astype(str).str.contains("(?i)Support", na=False, regex=True),
        hc_weekly_lob["LOB"].astype(str).str.contains("Nesting",     na=False, regex=True),
    ],
    ["Support", "Nesting"],
    default="Production"
)

hc_weekly_lob["LOB_Segment"] = hc_weekly_lob["LOB_Group"] + " " + hc_weekly_lob["LOB_Detail"]

last_12_weeks = sorted(hc_weekly_lob["Date Start Week"].dropna().unique())[-12:]
hc_weekly_lob = hc_weekly_lob[hc_weekly_lob["Date Start Week"].isin(last_12_weeks)]

def make_pivot(df, group_col):
    weekly = (
        df.groupby([group_col, "Date Start Week"], as_index=False)
          .agg(Agent_Count=("OracleID", "nunique"))
    )
    pivot = (
        weekly
        .pivot_table(index=group_col, columns="Date Start Week", values="Agent_Count", fill_value=0)
        .reset_index()
    )
    pivot.columns.name = None
    week_cols = pivot.columns[1:].tolist()

    # Grand Total row
    total_row = pd.DataFrame(
        [["Grand Total"] + pivot[week_cols].sum().tolist()],
        columns=pivot.columns
    )
    pivot = pd.concat([pivot, total_row], ignore_index=True)

    pivot.columns = (
        [pivot.columns[0]] +
        [f"WB_{pd.Timestamp(c).strftime('%y_%m_%d')}" for c in week_cols]
    )
    pivot = pivot.rename(columns={pivot.columns[0]: "LOB"})
    return pivot

hc_lob_pivot        = make_pivot(hc_weekly_lob, "LOB_Group")
hc_lob_detail_pivot = make_pivot(hc_weekly_lob, "LOB_Segment")

from IPython.display import display, HTML
display(HTML(
    f"<h4>Weekly HC by LOB — Last 12 Weeks</h4>{hc_lob_pivot.to_html(index=False)}"
    f"<br><h4>Weekly HC by LOB — Production / Nesting / Support (Last 12 Weeks)</h4>{hc_lob_detail_pivot.to_html(index=False)}"
))

LOB,WB_26_06_22,WB_26_06_29,WB_26_07_06,WB_26_07_13,WB_26_07_20,WB_26_07_27,WB_26_08_03,WB_26_08_10,WB_26_08_17,WB_26_08_24,WB_26_08_31,WB_26_09_07
LG,105.0,103.0,101.0,97.0,97.0,96.0,96.0,96.0,96.0,96.0,96.0,96.0
NL,28.0,28.0,25.0,24.0,22.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0
Grand Total,133.0,131.0,126.0,121.0,119.0,117.0,117.0,117.0,117.0,117.0,117.0,117.0
LOB,WB_26_06_22,WB_26_06_29,WB_26_07_06,WB_26_07_13,WB_26_07_20,WB_26_07_27,WB_26_08_03,WB_26_08_10,WB_26_08_17,WB_26_08_24,WB_26_08_31,WB_26_09_07
LG Nesting,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
LG Production,102.0,100.0,99.0,95.0,95.0,94.0,94.0,94.0,94.0,94.0,94.0,94.0
LG Support,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NL Nesting,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NL Production,16.0,28.0,25.0,24.0,22.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0
NL Support,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
tl_ids = [102029874, 102371964, 102477371, 103110013, 103117188, 103177016]

df_2026 = hc_extend[
    (hc_extend["Year"] == 2026)
    & (pd.to_numeric(hc_extend["TL ID"], errors="coerce").isin(tl_ids))
].copy()

df_2026["Date"]         = pd.to_datetime(df_2026["Date"],         errors="coerce")
df_2026["LWD/Movement"] = pd.to_datetime(df_2026["LWD/Movement"], errors="coerce")
df_2026["CCT Training"] = pd.to_datetime(df_2026["CCT Training"], errors="coerce")
df_2026["Month"]        = df_2026["Date"].dt.strftime("%y_%m")
df_2026["Quarter"]      = "Q" + df_2026["Date"].dt.quarter.astype("Int64").astype(str)
df_2026["LWD_Month"]    = df_2026["LWD/Movement"].dt.strftime("%y_%m")
df_2026["CCT_Month"]    = df_2026["CCT Training"].dt.strftime("%y_%m")
df_2026["LWD_Quarter"]  = np.where(
    df_2026["LWD/Movement"].notna(),
    "Q" + df_2026["LWD/Movement"].dt.quarter.astype("Int64").astype(str),
    np.nan
)
df_2026["CCT_Quarter"]  = np.where(
    df_2026["CCT Training"].notna(),
    "Q" + df_2026["CCT Training"].dt.quarter.astype("Int64").astype(str),
    np.nan
)

# 1 row per agent per TL for ATT / New Agent counting
df_agents = (
    df_2026
    .drop_duplicates(subset=["OracleID", "TL ID"])
    [["OracleID", "TL ID", "Supervisor Name",
      "LWD/Movement", "LWD_Month", "LWD_Quarter",
      "CCT Training", "CCT_Month", "CCT_Quarter"]]
)

# ATT Count
att_count_m = (
    df_agents[df_agents["LWD/Movement"].notna()]
    .groupby(["LWD_Month", "TL ID", "Supervisor Name"], as_index=False)
    .agg(ATT_Count=("OracleID", "nunique"))
    .rename(columns={"LWD_Month": "Month"})
)
att_count_q = (
    df_agents[df_agents["LWD/Movement"].notna()]
    .groupby(["LWD_Quarter", "TL ID", "Supervisor Name"], as_index=False)
    .agg(ATT_Count=("OracleID", "nunique"))
    .rename(columns={"LWD_Quarter": "Quarter"})
)

# New Agent Count
new_agent_m = (
    df_agents[df_agents["CCT Training"].notna()]
    .groupby(["CCT_Month", "TL ID", "Supervisor Name"], as_index=False)
    .agg(New_Agent_Count=("OracleID", "nunique"))
    .rename(columns={"CCT_Month": "Month"})
)
new_agent_q = (
    df_agents[df_agents["CCT Training"].notna()]
    .groupby(["CCT_Quarter", "TL ID", "Supervisor Name"], as_index=False)
    .agg(New_Agent_Count=("OracleID", "nunique"))
    .rename(columns={"CCT_Quarter": "Quarter"})
)

# HC Open/Closed — max per agent per period to avoid row duplication, then sum
hc_month = (
    df_2026
    .groupby(["Month", "TL ID", "Supervisor Name", "OracleID"], as_index=False)
    .agg(HC_Open_By_Month=("HC Open By Month", "max"), HC_Closed_By_Month=("HC Closed By Month", "max"))
    .groupby(["Month", "TL ID", "Supervisor Name"], as_index=False)
    .agg(HC_Open_By_Month=("HC_Open_By_Month", "sum"), HC_Closed_By_Month=("HC_Closed_By_Month", "sum"))
)
hc_quarter = (
    df_2026
    .groupby(["Quarter", "TL ID", "Supervisor Name", "OracleID"], as_index=False)
    .agg(HC_Open_By_Month=("HC Open By Month", "max"), HC_Closed_By_Month=("HC Closed By Month", "max"))
    .groupby(["Quarter", "TL ID", "Supervisor Name"], as_index=False)
    .agg(HC_Open_By_Month=("HC_Open_By_Month", "sum"), HC_Closed_By_Month=("HC_Closed_By_Month", "sum"))
)

def build_summary(hc_df, att_df, new_df, key, suffix="Month"):
    df = (
        hc_df
        .merge(att_df[[key, "TL ID", "ATT_Count"]],       on=[key, "TL ID"], how="left")
        .merge(new_df[[key, "TL ID", "New_Agent_Count"]], on=[key, "TL ID"], how="left")
    )
    df["ATT_Count"]       = df["ATT_Count"].fillna(0).astype(int)
    df["New_Agent_Count"] = df["New_Agent_Count"].fillna(0).astype(int)
    df["ATT %"] = (
        (df["ATT_Count"] / df["HC_Open_By_Month"] * 100)
        .replace([np.inf, np.nan], 0)
        .map("{:.1f}%".format)
    )
    return (
        df[[key, "TL ID", "Supervisor Name",
            "HC_Open_By_Month", "HC_Closed_By_Month",
            "New_Agent_Count", "ATT_Count", "ATT %"]]
        .rename(columns={
            "HC_Open_By_Month":   f"HC_Open_By_{suffix}",
            "HC_Closed_By_Month": f"HC_Closed_By_{suffix}",
        })
        .sort_values([key, "TL ID"])
        .reset_index(drop=True)
    )

def build_total(summary_df, key, suffix="Month"):
    open_col   = f"HC_Open_By_{suffix}"
    closed_col = f"HC_Closed_By_{suffix}"
    return (
        summary_df
        .groupby(key, as_index=False)
        .agg(**{
            open_col:           (open_col,          "sum"),
            closed_col:         (closed_col,        "sum"),
            "New_Agent_Count":  ("New_Agent_Count", "sum"),
            "ATT_Count":        ("ATT_Count",       "sum"),
        })
        .assign(**{"ATT %": lambda df: (df["ATT_Count"] / df[open_col] * 100).map("{:.1f}%".format)})
        .sort_values(key)
        .reset_index(drop=True)
    )

attrition_summary  = build_summary(hc_month,   att_count_m, new_agent_m, "Month",   suffix="Month")
attrition_by_month = build_total(attrition_summary,  "Month",   suffix="Month")
attrition_by_tl_q  = build_summary(hc_quarter, att_count_q, new_agent_q, "Quarter", suffix="Quarter")
attrition_by_q     = build_total(attrition_by_tl_q,  "Quarter", suffix="Quarter")

from IPython.display import display, HTML

html = f"""
<div style="display: flex; gap: 40px; align-items: flex-start; flex-wrap: wrap;">
    <div>
        <h4>By TL ID — Monthly</h4>
        {attrition_summary.to_html(index=False)}
    </div>
    <div>
        <h4>By Month</h4>
        {attrition_by_month.to_html(index=False)}
    </div>
    <div>
        <h4>By TL ID — Quarterly</h4>
        {attrition_by_tl_q.to_html(index=False)}
    </div>
    <div>
        <h4>By Quarter</h4>
        {attrition_by_q.to_html(index=False)}
    </div>
</div>
"""

display(HTML(html))

Month,TL ID,Supervisor Name,HC_Open_By_Month,HC_Closed_By_Month,New_Agent_Count,ATT_Count,ATT %
26_01,102029874,Truong Thien Thanh Toan,18,16,0,2,11.1%
26_01,102371964,Tran Thao Uyen,16,17,0,0,0.0%
26_01,102477371,Mia Minh Le,17,17,0,0,0.0%
26_01,103110013,Ann,16,16,0,0,0.0%
26_01,103117188,Chau Thien Kim,14,15,0,0,0.0%
26_02,102029874,Truong Thien Thanh Toan,15,15,0,0,0.0%
26_02,102371964,Tran Thao Uyen,17,17,0,0,0.0%
26_02,102477371,Mia Minh Le,17,17,0,0,0.0%
26_02,103110013,Ann,16,16,0,0,0.0%
26_02,103117188,Chau Thien Kim,15,15,0,0,0.0%


In [14]:
# 1/10 designation groups
desig_advisor = ["Advisor I, Customer Service"]
desig_sme     = ["SME, Operations", "Sr. SME, Operations"]
desig_tl_ops  = ["Supervisor, Business Operations", "Team Leader, Operations"]
desig_qa      = ["Sr. Quality Evaluator", "Quality Evaluator"]
desig_rta     = ["Sr. Representative, Real Time Management", "Associate, Real Time Management"]
desig_tl_wfm  = ["Supervisor, WFM"]
desig_trainer  = ["Communications Trainer II", "Communications Trainer I", "Trainer I"]
desig_leader   = [
    "Sr. Supervisor, WFM", "Supervisor, WFM", "Sr. Service Delivery Manager",
    "Supervisor, Training & Quality", "Manager II, Training & Quality",
    "Operations Manager I", "Supervisor, Business Operations",
    "Associate, Business Intelligence", "Associate Director, Operations Support",
    "Sr. Quality Evaluator", "Sr. Manager, Training & Quality",
    "Sr. Supervisor, Training & Quality", "Manager I, WFM",
    "Service Delivery Manager II", "Manager I, Communications Training",
    "Analyst, WFM Real Time Management",
]

# 2/10 filter to 2026 Mondays — ensure Date is datetime64 before weekday extraction
hc_extend["Date"] = pd.to_datetime(hc_extend["Date"], errors="coerce")
is_2026   = hc_extend["Year"].isin([2026])
is_monday = hc_extend["Date"].dt.weekday == 0
hc_monday = hc_extend[is_2026 & is_monday].copy().reset_index(drop=True)

# 3/10 boolean masks
stat_active = hc_monday["Status"] == "Active"
stat_termed = hc_monday["Status"] == "Terminated"
stat_trans  = hc_monday["Status"] == "Transfered"
dsg_adv = hc_monday["Designation"].isin(desig_advisor)
dsg_sme = hc_monday["Designation"].isin(desig_sme)
dsg_tlo = hc_monday["Designation"].isin(desig_tl_ops)
dsg_qa  = hc_monday["Designation"].isin(desig_qa)
dsg_rta = hc_monday["Designation"].isin(desig_rta)
dsg_tlw = hc_monday["Designation"].isin(desig_tl_wfm)
dsg_trn = hc_monday["Designation"].isin(desig_trainer)
dsg_ldr = hc_monday["Designation"].isin(desig_leader)
lob_supp_lg  = hc_monday["LOB"].str.contains("(?i)Support_LG",   na=False, regex=True)
lob_supp_nl  = hc_monday["LOB"].str.contains("(?i)Support_NL",   na=False, regex=True)
lob_flex_trn = hc_monday["LOB"].str.contains("(?i)Flex_Trainer",  na=False, regex=True)
lob_flex_qa  = hc_monday["LOB"].str.contains("(?i)Flex_QA",       na=False, regex=True)
lob_flex_sme = hc_monday["LOB"].str.contains("(?i)Flex_SME",      na=False, regex=True)
lob_nl       = hc_monday["LOB"].str.contains("Non_Lodging|NL",    na=False, regex=True)
lob_lg       = hc_monday["LOB"].str.contains("Lodging|LG",        na=False, regex=True)
lob_supp     = hc_monday["LOB"].str.contains("(?i)Support",       na=False, regex=True)
dtl_cct       = hc_monday["Detail Status"] == "CCT Training"
dtl_nl_trn    = hc_monday["Detail Status"] == "Non Lodging Training"
dtl_lg_trn    = hc_monday["Detail Status"] == "Lodging Training"
dtl_nesting   = hc_monday["Detail Status"].str.contains("(?i)Nesting",                  na=False, regex=True)
dtl_lg_renest = hc_monday["Detail Status"].str.contains("Lodging Extended Nesting",      na=False, regex=True)
dtl_nl_renest = hc_monday["Detail Status"].str.contains("Non Lodging Extended Nesting",  na=False, regex=True)

# 4/10 HC_File_Role via np.select — CCT and Extended Nesting before generic conditions
conditions = [
    dsg_adv & dtl_cct        & stat_active,
    dsg_adv & lob_supp_lg   & stat_active,
    dsg_adv & lob_supp_nl   & stat_active,
    dsg_adv & lob_flex_trn  & stat_active,
    dsg_adv & lob_flex_qa   & stat_active,
    dsg_adv & lob_flex_sme  & stat_active,
    dsg_adv & lob_nl & dtl_nl_trn  & stat_active,
    dsg_adv & dtl_lg_renest          & stat_active,
    dsg_adv & dtl_nl_renest          & stat_active,
    dsg_adv & lob_nl & dtl_nesting & stat_active,
    dsg_adv & lob_nl & stat_termed,
    dsg_adv & lob_nl & stat_trans,
    dsg_adv & lob_nl & stat_active,
    dsg_adv & lob_lg & dtl_lg_trn  & stat_active,
    dsg_adv & lob_lg & dtl_nesting & stat_active,
    dsg_adv & lob_lg & stat_termed,
    dsg_adv & lob_lg & stat_trans,
    dsg_adv & lob_lg & stat_active,
    dsg_sme & lob_nl & stat_active,
    dsg_sme & lob_nl & stat_termed,
    dsg_sme & lob_lg & stat_active,
    dsg_sme & lob_lg & stat_termed,
    dsg_sme & lob_supp & stat_active,
    dsg_sme & lob_supp & stat_termed,
    dsg_tlo & lob_nl & stat_active,
    dsg_tlo & lob_nl & stat_termed,
    dsg_tlo & lob_lg & stat_active,
    dsg_tlo & lob_lg & stat_termed,
    dsg_tlo & lob_supp & stat_active,
    dsg_tlo & lob_supp & stat_termed,
    stat_trans,
    dsg_tlo & stat_termed,
    dsg_qa  & stat_termed,
    dsg_rta & stat_termed,
    dsg_trn & stat_termed,
    dsg_qa  & stat_active,
    dsg_rta & stat_active,
    dsg_tlw & stat_active,
    dsg_ldr & stat_termed,
    dsg_ldr & stat_active,
]
choices = [
    "Training",
    "LG Chat Task", "NL Chat Task", "Trainer Task", "QA Task", "SME Task",
    "NL Chat Training",
    "LG Chat Renesting", "NL Chat Renesting",
    "NL Chat Nesting", "NL Chat Termed", "NL Chat Transferred", "NL Chat",
    "LG Chat Training", "LG Chat Nesting", "LG Chat Termed", "LG Chat Transferred", "LG Chat",
    "NL Chat SME", "NL Chat SME Termed", "LG Chat SME", "LG Chat SME Termed", "SME", "Termed",
    "NL Chat TL",  "NL Chat TL Termed",  "LG Chat TL",  "LG Chat Termed", "TL",  "Termed",
    "Transferred",
    "Retail TL Termed", "Retail QA Termed", "RTA Termed", "Retail Trainer Termed",
    "Retail QA", "RTA", "TL",
    "Leader Termed", "Leader",
]
hc_monday["HC_File_Role"] = np.select([c.fillna(False).astype(bool) for c in conditions], choices, default=None)

# 5/10 EWS integration — Active only, overwrite HC_File_Role on/after LWD Expected
ews = read_excel_data_only(folder_paths["hc_staffing"], "EWS")[["OracleID", "LOB", "LWD Expected", "WD"]].dropna(subset=["OracleID", "LWD Expected"])
ews["OracleID"]     = pd.to_numeric(ews["OracleID"], errors="coerce")
ews["LWD Expected"] = pd.to_datetime(ews["LWD Expected"], errors="coerce")
ews = ews.dropna(subset=["OracleID", "LWD Expected"])
ews = ews[ews["WD"].astype(str).str.upper().str.strip() == "ACTIVE"].drop(columns=["WD"])
print(f"EWS Active employees: {len(ews)}")
ews["EWS_Role"] = np.where(ews["LOB"].str.contains("Non_Lodging|NL", na=False, regex=True), "NL Chat Termed", "LG Chat Termed")
hc_monday["OracleID"] = pd.to_numeric(hc_monday["OracleID"], errors="coerce")
hc_monday["Date"]     = pd.to_datetime(hc_monday["Date"],    errors="coerce")
hc_monday = hc_monday.merge(ews[["OracleID", "LWD Expected", "EWS_Role"]], on="OracleID", how="left")
ews_mask  = hc_monday["LWD Expected"].notna() & (hc_monday["Date"] >= hc_monday["LWD Expected"])
hc_monday.loc[ews_mask, "HC_File_Role"] = hc_monday.loc[ews_mask, "EWS_Role"]
hc_monday = hc_monday.drop(columns=["LWD Expected", "EWS_Role"], errors="ignore")
print(f"EWS applied: {ews_mask.sum()} rows from {ews['OracleID'].nunique()} employees")

# 6/10 milestone columns per employee based on LOB
for _col in ["Lodging Training", "Lodging Nesting", "Lodging Nesting End", "Lodging Production",
             "NonLodging Training", "NonLodging Nesting", "NonLodging Nesting End", "NonLodging Production",
             "Contract Start Date", "Date"]:
    if _col in hc_monday.columns:
        hc_monday[_col] = pd.to_datetime(hc_monday[_col], errors="coerce")

_meta_src_cols = [c for c in [
    "OracleID", "Email Id", "Employee Name", "LOB",
    "Lodging Training", "Lodging Nesting", "Lodging Nesting End", "Lodging Production",
    "NonLodging Training", "NonLodging Nesting", "NonLodging Nesting End", "NonLodging Production",
    "Contract Start Date",
] if c in hc_monday.columns]

emp_meta = hc_monday.drop_duplicates(subset=["OracleID"])[_meta_src_cols].copy()

_is_lg = emp_meta["LOB"].str.contains("Lodging|LG", na=False, regex=True) & ~emp_meta["LOB"].str.contains("Non_Lodging|NL", na=False, regex=True)
_is_nl = emp_meta["LOB"].str.contains("Non_Lodging|NL", na=False, regex=True)

# 7/10 map LG/NL source cols to output milestone cols
_milestone_map = {
    "Training start date": ("Lodging Training",    "NonLodging Training"),
    "Training end date":   ("Lodging Nesting",     "NonLodging Nesting"),
    "Nesting start date":  ("Lodging Nesting",     "NonLodging Nesting"),
    "Nesting end date":    ("Lodging Nesting End", "NonLodging Nesting End"),
    "Production Date":     ("Lodging Production",  "NonLodging Production"),
}

for out_col, (lg_src, nl_src) in _milestone_map.items():
    lg_vals = emp_meta[lg_src] if lg_src in emp_meta.columns else pd.NaT
    nl_vals = emp_meta[nl_src] if nl_src in emp_meta.columns else pd.NaT
    emp_meta[out_col] = (
        pd.Series(np.where(_is_lg, lg_vals, np.where(_is_nl, nl_vals, pd.NaT)), index=emp_meta.index)
        .pipe(pd.to_datetime, errors="coerce")
        .dt.date
    )

if "Contract Start Date" in emp_meta.columns:
    emp_meta["Contract Start Date"] = pd.to_datetime(emp_meta["Contract Start Date"], errors="coerce").dt.date

_milestone_display = [c for c in ["Training start date", "Training end date","Nesting start date",  "Nesting end date",  "Contract Start Date", "Production Date"] if c in emp_meta.columns]

emp_meta = emp_meta[["OracleID"] + _milestone_display]

# 8/10 pivot one row per employee, one col per week
hc_pivot = pd.pivot_table(data=hc_monday, index=["OracleID", "Email Id", "Employee Name"], columns="Date Start Week", values="HC_File_Role", aggfunc="first").reset_index()
hc_pivot.columns.name = None

# 9/10 merge milestones after Employee Name
hc_pivot = hc_pivot.merge(emp_meta, on="OracleID", how="left")
_week_cols = [c for c in hc_pivot.columns if c not in ["OracleID", "Email Id", "Employee Name"] + _milestone_display]
hc_pivot = hc_pivot[["OracleID", "Email Id", "Employee Name"] + _milestone_display + _week_cols]

# 10/10 export
export_folder = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount"
hc_pivot.to_excel(f"{export_folder}/Pivot_HC_Role_2026.xlsx", index=False, engine="xlsxwriter")
hc_monday.to_excel(f"{export_folder}/HC_Role_2026.xlsx",      index=False, engine="xlsxwriter")
print("Exported Pivot_HC_Role_2026.xlsx and HC_Role_2026.xlsx")
hc_pivot

EWS Active employees: 4
EWS applied: 20 rows from 3 employees
Exported Pivot_HC_Role_2026.xlsx and HC_Role_2026.xlsx


,OracleID,Email Id,Employee Name,Training start date,Training end date,Nesting start date,Nesting end date,Contract Start Date,Production Date,2026-01-05 00:00:00,...,2026-07-06 00:00:00,2026-07-13 00:00:00,2026-07-20 00:00:00,2026-07-27 00:00:00,2026-08-03 00:00:00,2026-08-10 00:00:00,2026-08-17 00:00:00,2026-08-24 00:00:00,2026-08-31 00:00:00,2026-09-07 00:00:00
0,1081503,van.tran@concentrix.com,TRAN PHAM NGOC VAN,NaT,NaT,NaT,NaT,2019-05-24,NaT,Transferred,...,Transferred,Transferred,Transferred,Transferred,Transferred,Transferred,Transferred,Transferred,Transferred,Transferred
1,101606999,hoangminhduc.nguyen@concentrix.com,NGUYEN HOANG MINH DUC,NaT,NaT,NaT,NaT,2020-08-04,NaT,NaN,...,Leader Termed,Leader Termed,Leader Termed,Leader Termed,Leader Termed,Leader Termed,Leader Termed,Leader Termed,Leader Termed,Leader Termed
2,101885211,thixuantien.nguyen@concentrix.com,NGUYEN THI XUAN TIEN,NaT,NaT,NaT,NaT,2021-10-21,NaT,RTA Termed,...,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed
3,101957833,ducanh.nguyen@concentrix.com,NGUYEN DUC ANH,NaT,NaT,NaT,NaT,2022-02-10,NaT,Termed,...,Termed,Termed,Termed,Termed,Termed,Termed,Termed,Termed,Termed,Termed
4,101973765,sanghdok.thanh@concentrix.com,THANH SANG HDOK,2024-03-04,2024-03-25,2024-03-25,2024-04-06,2022-03-07,NaT,LG Chat Termed,...,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
879,103611143,quockhanh.doan@concentrix.com,DOAN QUOC KHANH,NaT,NaT,NaT,NaT,2026-06-15,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
880,103611144,phunganhtu.tran@concentrix.com,TRAN PHUNG ANH TU,NaT,NaT,NaT,NaT,2026-06-15,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
881,103611150,thanhphat.chau@concentrix.com,CHAU THANH PHAT,NaT,NaT,NaT,NaT,2026-06-15,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
882,103611151,thianhthu.nguyen13@concentrix.com,NGUYEN THI ANH THU,NaT,NaT,NaT,NaT,2026-06-15,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
base_filter = (
    hc_extend["Designation"].astype(str).str.contains("Advisor I, Customer Service", na=False)
    & (hc_extend["Status"].astype(str) == "Active")
    & (hc_extend["Date"].dt.year >= 2026)
)

lg_active = (
    hc_extend[
        base_filter
        & hc_extend["Detail Status"].astype(str).isin(["Lodging Production", "Lodging Nesting"])
    ]
    .groupby(["Month", "Date"], as_index=False)
    .agg(LG_Active=("OracleID", "nunique"))
)

nl_active = (
    hc_extend[
        base_filter
        & hc_extend["Detail Status"].astype(str).isin(["Non Lodging Production", "Non Lodging Nesting"])
    ]
    .groupby(["Month", "Date"], as_index=False)
    .agg(NL_Active=("OracleID", "nunique"))
)

hc_target_plan = pd.merge(lg_active, nl_active, on=["Month", "Date"], how="outer").fillna(0)
hc_target_plan["LG Active"] = hc_target_plan["LG_Active"].astype(int)
hc_target_plan["NL Active"] = hc_target_plan["NL_Active"].astype(int)
hc_target_plan = hc_target_plan.drop(columns=["LG_Active", "NL_Active"])

ATT_TARGET_RATE = 0.06
hc_target_plan["Target Planned%"] = "6%"
hc_target_plan["LG Planned"] = (hc_target_plan["LG Active"] * ATT_TARGET_RATE).round().astype(int)
hc_target_plan["NL Planned"] = (hc_target_plan["NL Active"] * ATT_TARGET_RATE).round().astype(int)

hc_target_plan = hc_target_plan.sort_values("Date").reset_index(drop=True)

export_path = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/Advisor Headcount ATT Plan Daily.csv"
hc_target_plan.to_csv(export_path, index=False)
print(f"Exported: {export_path}")

hc_target_plan

Exported: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/Advisor Headcount ATT Plan Daily.csv


,Month,Date,LG Active,NL Active,Target Planned%,LG Planned,NL Planned
0,Jan-26,2026-01-01,94,0,6%,6,0
1,Jan-26,2026-01-02,94,0,6%,6,0
2,Jan-26,2026-01-03,94,0,6%,6,0
3,Jan-26,2026-01-04,94,0,6%,6,0
4,Jan-26,2026-01-05,93,0,6%,6,0
...,...,...,...,...,...,...,...
246,Sep-26,2026-09-04,101,33,6%,6,2
247,Sep-26,2026-09-05,101,33,6%,6,2
248,Sep-26,2026-09-06,101,33,6%,6,2
249,Sep-26,2026-09-07,101,33,6%,6,2


In [16]:
# 1/5 ensure date cols are datetime
hc_extend["Date"]         = pd.to_datetime(hc_extend["Date"],         errors="coerce")
hc_extend["LWD/Movement"] = pd.to_datetime(hc_extend["LWD/Movement"], errors="coerce")

# 2/5 group by Month and export one CSV per month
for month_label, group in hc_extend.groupby("Month"):
    sample_date    = pd.to_datetime(group["Date"].dropna().iloc[0], errors="coerce")
    snapshot_month = sample_date.to_period("M")

    # 3/5 compute months since LWD
    lwd_period = group["LWD/Movement"].dt.to_period("M")
    months_since_lwd = (snapshot_month.year - lwd_period.dt.year) * 12 + (snapshot_month.month - lwd_period.dt.month)

    # 4/5 drop stale exits (termed/transferred > 3 months ago)
    is_exited = group["LWD/Movement"].notna()
    is_stale  = months_since_lwd > 3
    group_filtered = group[~(is_exited & is_stale)].copy()

    # 5/5 write CSV
    dropped   = len(group) - len(group_filtered)
    file_name = sample_date.strftime("%y_%m") + ".csv"
    file_path = os.path.join(folder_paths["hc_extend_by_month"], file_name)
    group_filtered.to_csv(file_path, index=False)
    print(f"Written: {file_name}  |  rows: {len(group_filtered):,}  |  dropped: {dropped:,}")


Written: 24_04.csv  |  rows: 8,835  |  dropped: 630
Written: 25_04.csv  |  rows: 15,017  |  dropped: 7,770
Written: 26_04.csv  |  rows: 4,443  |  dropped: 20,310
Written: 24_08.csv  |  rows: 8,990  |  dropped: 2,542
Written: 25_08.csv  |  rows: 9,207  |  dropped: 15,314
Written: 26_08.csv  |  rows: 5,766  |  dropped: 22,010
Written: 23_12.csv  |  rows: 3,591  |  dropped: 0
Written: 24_12.csv  |  rows: 13,886  |  dropped: 4,743
Written: 25_12.csv  |  rows: 6,014  |  dropped: 18,538
Written: 24_02.csv  |  rows: 6,672  |  dropped: 0
Written: 25_02.csv  |  rows: 15,214  |  dropped: 5,516
Written: 26_02.csv  |  rows: 3,820  |  dropped: 18,396
Written: 24_01.csv  |  rows: 6,047  |  dropped: 0
Written: 25_01.csv  |  rows: 16,825  |  dropped: 5,363
Written: 26_01.csv  |  rows: 4,495  |  dropped: 20,057
Written: 24_07.csv  |  rows: 9,001  |  dropped: 2,201
Written: 25_07.csv  |  rows: 10,557  |  dropped: 13,950
Written: 26_07.csv  |  rows: 6,169  |  dropped: 21,607
Written: 24_06.csv  |  rows: 

In [17]:
output_folder = folder_paths["team_alignment_wow"]
os.makedirs(output_folder, exist_ok=True)

hc_monday["Date"]            = pd.to_datetime(hc_monday["Date"],            errors="coerce")
hc_monday["Date Start Week"] = pd.to_datetime(hc_monday["Date Start Week"], errors="coerce")

df_align = hc_monday[
    hc_monday["Designation"].astype(str).str.contains("Advisor I, Customer Service", na=False)
    & hc_monday["HC_File_Role"].notna()
    & (hc_monday["HC_File_Role"].astype(str).str.strip() != "")
    & ~hc_monday["HC_File_Role"].astype(str).str.contains("Termed|Transferred", na=False)
].copy()

df_align["LOB_Combine"] = np.select(
    [   
        df_align["LOB"].str.contains("(?i)Support",    na=False, regex=True),
        df_align["LOB"].str.contains("Non_Lodging|NL", na=False, regex=True),
        df_align["LOB"].str.contains("Lodging|LG",     na=False, regex=True),
    ],
    ["Support", "Non_Lodging", "Lodging"],
    default=None,
)

df_align = (
    df_align
    .sort_values("Date", ascending=False)
    .drop_duplicates(subset=["OracleID", "Date Start Week"])
)

export_cols = [c for c in [
    "IEX ID", "OracleID", "Employee Name", "Email Id",
    "TL ID", "Supervisor Name", "Designation", "LOB", "LOB_Combine", "Site",
    "HC_File_Role", "Date Start Week",
] if c in df_align.columns]

for week_start, group in df_align.groupby("Date Start Week"):
    if pd.isna(week_start):
        continue
    week_str  = pd.Timestamp(week_start).strftime("%Y_%m_%d")
    file_name = f"Team_Alignment_{week_str}.csv"
    file_path = os.path.join(output_folder, file_name)
    group[export_cols].to_csv(file_path, index=False)
    print(f"Written: {file_name}  |  agents: {len(group)}")

print("Done.")

Written: Team_Alignment_2026_01_05.csv  |  agents: 93
Written: Team_Alignment_2026_01_12.csv  |  agents: 93
Written: Team_Alignment_2026_01_19.csv  |  agents: 93
Written: Team_Alignment_2026_01_26.csv  |  agents: 93
Written: Team_Alignment_2026_02_02.csv  |  agents: 92
Written: Team_Alignment_2026_02_09.csv  |  agents: 92
Written: Team_Alignment_2026_02_16.csv  |  agents: 92
Written: Team_Alignment_2026_02_23.csv  |  agents: 92
Written: Team_Alignment_2026_03_02.csv  |  agents: 91
Written: Team_Alignment_2026_03_09.csv  |  agents: 90
Written: Team_Alignment_2026_03_16.csv  |  agents: 89
Written: Team_Alignment_2026_03_23.csv  |  agents: 82
Written: Team_Alignment_2026_03_30.csv  |  agents: 96
Written: Team_Alignment_2026_04_06.csv  |  agents: 98
Written: Team_Alignment_2026_04_13.csv  |  agents: 95
Written: Team_Alignment_2026_04_20.csv  |  agents: 110
Written: Team_Alignment_2026_04_27.csv  |  agents: 108
Written: Team_Alignment_2026_05_04.csv  |  agents: 124
Written: Team_Alignment_2